

# Install



In [3]:
%%capture
!pip install unsloth
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf

#  Load dynamic 4-bit model

In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit",
    max_seq_length = 4096,   # max prompt+answer length; keep modest to save VRAM
    dtype = None,            # auto-detect
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)   # turns on fast inference mode

print(f"VRAM used after loading: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:2216: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.6: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


VRAM used after loading: 2.21 GB


# Documents

In [5]:
import os
os.makedirs("docs", exist_ok=True)

open("docs/hypertension.txt","w").write("""
Hypertension (High Blood Pressure).
Hypertension is a condition in which the force of blood against the artery walls is persistently too high.
In adults, a reading of 130/80 mmHg or higher is generally considered high, and 120/80 mmHg or lower is considered normal.
Hypertension often has no symptoms, which is why it is called the silent killer. Severe cases may cause headaches, shortness of breath or nosebleeds.
Risk factors include a high-salt diet, obesity, physical inactivity, excessive alcohol intake, smoking, stress, older age and family history.
Untreated hypertension increases the risk of heart attack, stroke, kidney disease and vision loss.
Management includes reducing salt, regular exercise, maintaining a healthy weight, limiting alcohol, stopping smoking and taking prescribed medication.
""")

open("docs/type2_diabetes.txt","w").write("""
Type 2 Diabetes Mellitus.
Type 2 diabetes is a chronic condition in which the body becomes resistant to insulin or does not produce enough insulin, causing high blood sugar.
Common symptoms include increased thirst, frequent urination, fatigue, blurred vision, slow-healing wounds and unexplained weight loss.
A fasting blood glucose of 126 mg/dL or higher on two tests, or an HbA1c of 6.5 percent or higher, indicates diabetes.
Risk factors include overweight, physical inactivity, family history, age over 45 and a history of gestational diabetes.
Long-term complications include heart disease, kidney damage (nephropathy), nerve damage (neuropathy), eye damage (retinopathy) and foot problems.
Management includes a balanced diet, regular physical activity, weight control, blood sugar monitoring and medications such as metformin or insulin as prescribed by a doctor.
""")

open("docs/asthma.txt","w").write("""
Asthma.
Asthma is a chronic lung disease in which the airways become inflamed and narrowed, making breathing difficult.
Typical symptoms are wheezing, coughing (especially at night or early morning), chest tightness and shortness of breath.
Common triggers include dust mites, pollen, pet dander, smoke, cold air, exercise, respiratory infections and strong odors.
Treatment usually includes quick-relief inhalers (bronchodilators) for sudden symptoms and long-term controller medicines such as inhaled corticosteroids.
An asthma attack that does not improve with a rescue inhaler, or causes difficulty speaking, blue lips or severe breathlessness, is an emergency and needs immediate medical care.
Avoiding triggers and following an asthma action plan from a doctor help prevent attacks.
""")

open("docs/cold_vs_flu.txt","w").write("""
Common Cold versus Influenza (Flu).
Both are viral respiratory infections, but influenza is usually more severe than the common cold.
Common cold symptoms develop gradually and include a runny nose, sneezing, sore throat and mild cough. Fever is uncommon or mild.
Influenza symptoms start suddenly and include high fever, body aches, chills, extreme tiredness, headache and dry cough.
Antibiotics do not work against viral infections such as colds and flu. Treatment focuses on rest, fluids and symptom relief.
Annual flu vaccination is recommended to reduce the risk of influenza and its complications such as pneumonia.
Seek medical care if there is difficulty breathing, chest pain, persistent high fever, confusion or if symptoms improve and then suddenly worsen.
""")

open("docs/anemia.txt","w").write("""
Iron Deficiency Anemia.
Anemia is a condition in which the blood has fewer red blood cells or less hemoglobin than normal, reducing oxygen delivery to the body.
Iron deficiency is the most common cause of anemia worldwide.
Symptoms include fatigue, weakness, pale skin, dizziness, shortness of breath, cold hands and feet and brittle nails.
Causes include poor dietary iron intake, blood loss such as heavy menstrual periods or gastrointestinal bleeding, pregnancy and poor iron absorption.
Diagnosis is made with blood tests including hemoglobin, ferritin and a complete blood count.
Iron-rich foods include red meat, lentils, spinach, beans and fortified cereals. Vitamin C helps the body absorb iron.
Treatment depends on the cause and should be directed by a healthcare professional.
""")

print(os.listdir("docs"))

['asthma.txt', 'cold_vs_flu.txt', 'anemia.txt', 'hypertension.txt', 'type2_diabetes.txt']


# Chunking

In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

txt_docs = DirectoryLoader("docs", glob="**/*.txt", loader_cls=TextLoader).load()
pdf_docs = DirectoryLoader("docs", glob="**/*.pdf", loader_cls=PyPDFLoader).load()
documents = txt_docs + pdf_docs

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"{len(documents)} documents -> {len(chunks)} chunks")
print(chunks[0].page_content)

/tmp/ipykernel_402/3221001658.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader


5 documents -> 10 chunks
Asthma.
Asthma is a chronic lung disease in which the airways become inflamed and narrowed, making breathing difficult.
Typical symptoms are wheezing, coughing (especially at night or early morning), chest tightness and shortness of breath.
Common triggers include dust mites, pollen, pet dander, smoke, cold air, exercise, respiratory infections and strong odors.


# Embeddings and index

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device": "cpu"},
)

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")          # optional: save so you can reload it later

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})   # fetch the top 3 chunks

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
for d in retriever.invoke("What are the symptoms of type 2 diabetes?"):
    print("-", d.page_content[:150].replace("\n"," "), "\n")

- Type 2 Diabetes Mellitus. Type 2 diabetes is a chronic condition in which the body becomes resistant to insulin or does not produce enough insulin, ca 

- Risk factors include overweight, physical inactivity, family history, age over 45 and a history of gestational diabetes. Long-term complications inclu 

- Common Cold versus Influenza (Flu). Both are viral respiratory infections, but influenza is usually more severe than the common cold. Common cold symp 



In [9]:
for q in ["What blood pressure is considered high?", "Which foods contain iron?"]:
    print("Q:", q)
    for d in retriever.invoke(q):
        print("  -", d.page_content[:100].replace("\n"," "))
    print()

Q: What blood pressure is considered high?
  - Hypertension (High Blood Pressure). Hypertension is a condition in which the force of blood against 
  - Risk factors include a high-salt diet, obesity, physical inactivity, excessive alcohol intake, smoki
  - Type 2 Diabetes Mellitus. Type 2 diabetes is a chronic condition in which the body becomes resistant

Q: Which foods contain iron?
  - Diagnosis is made with blood tests including hemoglobin, ferritin and a complete blood count. Iron-r
  - Iron Deficiency Anemia. Anemia is a condition in which the blood has fewer red blood cells or less h
  - Risk factors include a high-salt diet, obesity, physical inactivity, excessive alcohol intake, smoki



# RAG function

In [10]:
def rag_answer(question, k=2, max_new_tokens=256, show_sources=True):
    # 1) RETRIEVE (k now actually controls how many chunks are fetched)
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))

    # 2) BUILD A GROUNDED PROMPT
    messages = [
        {"role": "system", "content":
          "You are a medical information assistant for educational purposes only. "
          "Answer ONLY using the provided context. "
          "If the answer is not in the context, say: 'I don't know based on the provided documents.' "
          "Do not give personal diagnoses or drug doses. "
          "Be concise and end with: 'This is educational information, not medical advice.'"},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    # 3) GENERATE
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True,
            use_cache=True,
        )
    answer = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

    if show_sources:
        print("SOURCES:", [d.metadata.get("source") for d in docs])
    return answer

# Testing

In [11]:
questions = [
    "What are the symptoms of type 2 diabetes?",
    "What blood pressure reading counts as high?",
    "What should I do if my rescue inhaler doesn't help an asthma attack?",
    "What is the difference between a cold and the flu?",
    "Which foods are rich in iron?",
    "What is the treatment for malaria?",   # NOT in the documents -> should say it doesn't know
]
for q in questions:
    print("Q:", q)
    print("A:", rag_answer(q), "\n" + "-"*60)

Q: What are the symptoms of type 2 diabetes?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SOURCES: ['docs/type2_diabetes.txt', 'docs/type2_diabetes.txt']
A: According to the provided context, the common symptoms of Type 2 Diabetes Mellitus include:

1. Increased thirst
2. Frequent urination
3. Fatigue
4. Blurred vision
5. Slow-healing wounds
6. Unexplained weight loss

This is educational information, not medical advice. 
------------------------------------------------------------
Q: What blood pressure reading counts as high?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SOURCES: ['docs/hypertension.txt', 'docs/hypertension.txt']
A: In adults, a reading of 130/80 mmHg or higher is generally considered high. 
------------------------------------------------------------
Q: What should I do if my rescue inhaler doesn't help an asthma attack?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SOURCES: ['docs/asthma.txt', 'docs/asthma.txt']
A: If your rescue inhaler doesn't help an asthma attack, seek immediate medical care. This is because the symptoms have worsened and may be life-threatening. 
------------------------------------------------------------
Q: What is the difference between a cold and the flu?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SOURCES: ['docs/cold_vs_flu.txt', 'docs/cold_vs_flu.txt']
A: The main difference between a cold and the flu is the severity and onset of symptoms. A cold develops gradually, while the flu starts suddenly. Additionally, the flu typically includes symptoms such as high fever, body aches, chills, and extreme tiredness, which are not usually present in a cold.

This is educational information, not medical advice. 
------------------------------------------------------------
Q: Which foods are rich in iron?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SOURCES: ['docs/anemia.txt', 'docs/anemia.txt']
A: Red meat, lentils, spinach, beans, and fortified cereals are iron-rich foods. 
------------------------------------------------------------
Q: What is the treatment for malaria?
SOURCES: ['docs/asthma.txt', 'docs/cold_vs_flu.txt']
A: I don't know based on the provided documents. 

This is educational information, not medical advice. 
------------------------------------------------------------


# VRAM

In [12]:
print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
print(f"Reserved VRAM: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

Peak VRAM: 2.37 GB
Reserved VRAM: 2.41 GB


# Comparison

In [13]:
q = "What blood pressure reading counts as high hypertension, and what are the risk factors?"

# WITHOUT RAG (model's own memory only)
msgs = [{"role":"user","content":q}]
x = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
o = model.generate(input_ids=x, max_new_tokens=300, use_cache=True)
print("WITHOUT RAG:\n", tokenizer.decode(o[0][x.shape[1]:], skip_special_tokens=True))

# WITH RAG (answers from your documents)
print("\nWITH RAG:\n", rag_answer(q))

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITHOUT RAG:
 In adults, the American Heart Association (AHA) defines high blood pressure as a reading of 130/80 mmHg or higher. This is based on the SPRINT trial, which found that a systolic blood pressure (top number) of 130 mmHg or higher was associated with an increased risk of cardiovascular events.

Here's a breakdown of the blood pressure categories:

* Normal: Less than 120/80 mmHg
* Elevated: 120-129/80 mmHg
* Hypertension Stage 1: 130-139/80-89 mmHg
* Hypertension Stage 2: 140 or higher/90 or higher mmHg

Risk factors for high blood pressure include:

1. Age: The risk of high blood pressure increases with age, especially after 40.
2. Family history: Having a family history of high blood pressure can increase your risk.
3. Obesity: Being overweight or obese can increase your risk of high blood pressure.
4. Physical inactivity: A sedentary lifestyle can contribute to high blood pressure.
5. Smoking: Smoking can damage blood vessels and increase blood pressure.
6. High sodium in

In [14]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))

def generate(inputs):
    messages = [
        {"role": "system", "content":
            "You are a medical information assistant for educational purposes only. "
            "Answer ONLY using the provided context. "
            "If the answer is not in the context, say: 'I don't know based on the provided documents.' "
            "Do not give personal diagnoses or drug doses. "
            "Be concise and end with: 'This is educational information, not medical advice.'"},
        {"role": "user", "content": f"Context:\n{inputs['context']}\n\nQuestion: {inputs['question']}"},
    ]
    x = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        o = model.generate(input_ids=x, max_new_tokens=256, temperature=0.2, do_sample=True, use_cache=True)
    return tokenizer.decode(o[0][x.shape[1]:], skip_special_tokens=True)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RunnableLambda(generate)
)

print(rag_chain.invoke("What are the symptoms of type 2 diabetes?"))

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


According to the provided context, the symptoms of Type 2 Diabetes Mellitus include:

1. Increased thirst
2. Frequent urination
3. Fatigue
4. Blurred vision
5. Slow-healing wounds
6. Unexplained weight loss

This is educational information, not medical advice.
